# **Project title:** Quantitative assessment of slc35a2 localization to the Golgi in single cells
### **Researcher:** Xavier Williams
### **Notebook author:** Shannon Rhoads (srhoads@unc.edu)

### **Editing & version log:**
- v0.1: 20260416 - downloaded [batch_process_segmentations.ipynb](https://github.com/SCohenLab/infer-subc/blob/main/notebooks/part_1_segmentation_workflows/batch_process_segmentations.ipynb) from [infer-subc version v2.0.0b1](https://github.com/SCohenLab/infer-subc/tree/v2.0.0b1)
- v0.2: 20260416 - the following edits were made:
    - Remove all workflows except Golgi and Masks
    - Updated file reader to bioio (required new meta_dict creation for proper export of .tif segmentation files)


----------
## **How to use Jupyter notebooks:**

Advance through each block of code below by pressing `Shift`+`Enter` or pressing the "Execute Cell" (`▶️`) button to the left of each block.

You will see a series of instructions before each block of code. Be on the look out for the following headers and follow the instructions accordingly:
- &#x1F3C3; **Run code; no user input required** - proceed without adding anything to the code block
- &#x1F453; **FYI** (for your information) - helpful information usually to bring context to what is going on
- &#x1F6D1; &#x270D; **User Input Required** - stop and input the appropriate information about your data. The following indicator will also be present in the code block:
   ```python 
   #### USER INPUT REQUIRED ###
   ```

----------

# **Notebook title:** Batch Process Segmentations

***Prior to this notebook, you should have already run through notebooks 1.1 and 1.4 to determined the best segmetnation settings for the Golgi and masks.***

### **Purpose**
In this notebook, the settings determined in notebooks 1.1 and 1.4 will be used to batch process segmentations for multiple images. Images from a single file folder can be analyzed simultaneously with one or up to all of the segmentation workflows simultaneously. The output files are saved into a single file folder in preparation for quality assessement and quantitative analysis.


### ➡️ **Input**
The following files and information will be necessary for batch processing segmentations:
1. The **raw confocal microscopy images** (".tiff", ".tif", or ".czi") from one experimental replicate, where each channel of the image represents one of the organelles or structures being segmented. The pipeline will only accept a single file path as the input for the raw data and all images of the same type within that folder will be processed.
2. The **segementation settings** for each workflow you wish to include in the batch processing (*Hint: these are deteremined in notebooks 1.1 and 1.4*). This information is typed manually into the functions below (or copy and pasted from the print statement at the end of the individual notebooks).
3. The **output file location** where the resulting segmentation files will be saved.


### **Output** ➡️
One ".tiff" segmentation file will be output per segmentation workflow per image. They will all be saved into the same output folder. For proper quantitative analysis, all of the segmentation files for a single experimental replicate should be saved in the same folder. The naming convention for the segmentation outputs is to append the organelle or masks "nickname" of that structure to the end of the raw file name. This is specified for you below (e.g., "...-golgi.tiff" or "...-masks.tiff").

> #### **Example file structure:**
>
> - 📂 **Experiment1**
>   - 📂 raw-data               ```<-- Input directory```
>       - 🗋 "image1.tiff"
>       - 🗋 ...
>   - 📂 segmentation-output               ```<-- Output directory```
>       - 🗋 "image1-masks.tiff"
>       - 🗋 "image1-golgi.tiff"
>       - 🗋 ...
> - 📂 **Experiment2**
>   - 📂 ...
>
-----

### 👣 **Summary of steps**  

➡️ **BATCH PROCESSING**
- **`Step 1`** - Define the paths to your data and desired output folder.

- **`Step 2`** - Define the settings for each workflow being used.

    Workflow options include:
    - Infer [`masks`](./1.1_infer_masks_from-composite_with_nuc.ipynb)
    - Infer [`golgi`](1.4_infer_golgi.ipynb)

    *The desired settings for each workflow should be determined in notebooks 1.1-1.8 before beginning batch processing.*

- **`Step 3`** - Run `batch_process_segmentation()`
    - **Input**: Raw images
    - **Output**: Segmentation images

---------------------
## **IMPORTS**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block imports the Python packages needed in the batch processing steps below.

In [1]:
from pathlib import Path
from typing import List, Union
import os
import time

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files)
from infer_subc.organelles import *

import sys
sys.path.insert(0, str(Path("..").resolve()))
from src.segmentation import batch_process_segmentation

-----

## **BATCH PROCESSING**

### **`STEP 1` - Define the paths to your data**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `raw_path`: A string or a Path object of the path to your raw intensity images that will be the input for segmentation
- `raw_file_type`: The raw file type (e.g., ".tiff" or ".czi")
- `seg_path`: A string or a Path object of the path where the segmentation outputs will be saved
- `name_suffix`: An optional string to include before the segmentation suffix at the end of the output file. For example, if the name_suffix was "20240105", the segmentation file output from the 1.1_masks workflow would include: "{base-file-name}-20240105-masks".

In [2]:
#### USER INPUT REQUIRED IF NOT USING SAMPLE DATA ###
raw_path = Path(os.path.dirname(os.getcwd()))  / "pilot-data/single"
raw_file_type = ".czi"
seg_path = Path(os.path.dirname(os.getcwd()))  / "test-segmentation/single"
name_suffix = None

### **`STEP 2` - Define the settings for each workflow being used**

#### &#x1F6D1; &#x270D; **User Input Required:**

For each workflow that you wish to include in the batch processing, please fill out the information in the associated settings list. If you don't want to include a specific segmentation workflow, specify `None` instead.

The necessary settings for each function can be copied as a list from the end of the [mask](/analysis-pipelines/1.1_infer_masks_from-composite_with_nuc.ipynb) or [Golgi](/analysis-pipelines/1.4_infer_golgi.ipynb) workflow notebooks.

In [3]:
#### USER INPUT REQUIRED ###
masks_settings = None
golgi_settings = [0,
 4,
 1.4,
 'median',
 1.3,
 1200,
 5,
 2,
 0.05,
 0.65,
 0,
 0,
 0,
 0,
 '3D',
 0,
 0,
 2,
 '3D',
 True,
 1.0,
 1,
 False,
 1.0,
 0]

### **`STEP 3` - Run batch processing**

#### &#x1F3C3; **Run code; no user input required**
&#x1F453; **FYI:** This code block uses the inputs provided above to run the batch processing for raw images. The output of the cell will state when each image is completed.

In [4]:
batch = batch_process_segmentation(raw_path,
                                   raw_file_type,
                                   seg_path,
                                   name_suffix,
                                   masks_settings,
                                   golgi_settings)

The specified 'seg_path' was not found. Creating c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-segmentation\single.
Beginning segmentation of: c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\single\03022026R55LB1T1P1-01.czi


Failed to parse position index from scene name 'Image:0': invalid literal for int() with base 10: 'mage:0'
Traceback (most recent call last):
  File "c:\Users\Shannon\anaconda3\envs\golgi-coloc\lib\site-packages\bioio_czi\standard_metadata.py", line 20, in position_index
    return int(prefix[1:])
ValueError: invalid literal for int() with base 10: 'mage:0'


Processing for c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\pilot-data\single\03022026R55LB1T1P1-01.czi completed in 2.7221044937769574 minutes.
Batch processing complete: 1 images segmented in 2.7221044937769574 minutes.


-------------
### ✅ **BATCH PROCESSING SEGMENTATIONS COMPLETE!** 

### 🔎 **Next steps - Quality Check (QC) and Mask Separation**
After segmenting all the cells in your dataset, you will continue on to the [quality_check_segmentations.ipynb](quality_check_segmentations.ipynb) notebook. 

*The QC notebook will allow you tooOpen each image and the matching segmentation outputs in Napari for: <ins>visual inspection</ins>*
